# Attention sinks, sliding windows and the learned sink logit

*Xiao et al., "Efficient Streaming Language Models with Attention Sinks" (StreamingLLM), ICLR 2024 ([arXiv 2309.17453](https://arxiv.org/abs/2309.17453)); the learned sink logit used by gpt-oss and MiMo-V2-Flash; the local/global layer mix of Gemma 3 and Trinity. Notebook 3 of 9 in the "How LLMs build on each other" series.*

**TL;DR.** A chat that never ends cannot keep every earlier token in the KV cache. The obvious fix is a **sliding window**: keep only the last $w$ tokens. Xiao et al. found that this breaks the model completely the moment the *first* token falls out of the window, even though that token carries no useful meaning. The reason is the softmax: attention weights must add up to 1, so a head with nothing to say still has to put its weight somewhere, and trained models learn to park it on the first tokens, which are visible to everyone. Take those away and every other weight is inflated. Two fixes: keep a few initial tokens in the cache forever (StreamingLLM), or give the softmax an explicit place to dump weight, a **learned sink logit** per head (gpt-oss). A third, structural fix is to make most layers local and only a few global (Gemma 3, Trinity).

This notebook rebuilds each piece, shows the arithmetic of why eviction breaks softmax, trains small models with and without sink mechanisms, and measures streaming loss with each cache policy.

| step | what we do |
|---|---|
| 1 | the streaming problem and the paper's Figure 1 |
| 2 | why softmax needs a sink: the arithmetic |
| 3 | the StreamingLLM cache: sinks plus window, and RoPE positions inside the cache |
| 4 | the learned sink logit and softmax-off-by-one |
| 5 | a controlled case: train a head that needs a sink, then evict it |
| 6 | experiment: three kinds of toy language model, five cache policies |
| 7 | local and global layers: cache and reach |

In [1]:
import sys, math, time
from pathlib import Path
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path.cwd()))             # the notebook lives next to dsv3kit.py
import archkit as A                             # tested building blocks; this notebook re-shows the code it uses

torch.manual_seed(0)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": .25, "axes.spines.top": False,
                     "axes.spines.right": False, "font.size": 10})
C = dict(blue="#1f78b4", orange="#d95f02", green="#1b9e77", purple="#7570b3", red="#c0392b", grey="#777777")

PASS, FAIL = [], []
def check(name, ok, detail=""):
    (PASS if ok else FAIL).append(name)
    print(("PASS" if ok else "FAIL"), "-", name, detail)

print("torch", torch.__version__)

torch 2.14.1


## 1. The streaming problem

A chat assistant, a coding agent or a log monitor reads a stream that is much longer than any training sequence. Dense attention (keep everything) costs $O(T^2)$ time, a cache that keeps growing, and, because models are trained at a finite length, it degrades past the training length. **Window attention** (keep the last $L$ tokens) fixes the memory and the cost, $O(TL)$, but the paper shows that it **collapses** as soon as the text is longer than the cache. **Sliding window with re-computation** (rebuild the last $L$ tokens from scratch for every new token) works but costs $O(TL^2)$. StreamingLLM keeps a handful of initial tokens plus the window and gets the quality of the re-computation baseline at the cost of window attention.

![The paper's Figure 1, for Llama-2-13B on a 65K-token book. (a) dense attention: O(T^2), perplexity 5641. (b) window attention: perplexity 5158, it "breaks when initial tokens are evicted". (c) sliding window with re-computation: perplexity 5.43 but slow. (d) StreamingLLM: perplexity 5.40 at the cost of (b).](../../assets/architecture_gallery/sink_fig1_streamingllm.png)

Perplexity 5.4 against 5158 is the gap to explain. The paper shows it is not semantics: replacing the first four tokens by line-break tokens gives perplexity 5.60 (their Table 1), nearly as good. The *content* of the initial tokens does not matter; their *presence* does.

## 2. Why softmax needs a sink

Attention weights for one query are $a_j = e^{x_j}/\sum_i e^{x_i}$: positive and summing to 1. Suppose a head has learned to do something only occasionally, say "look for a closing bracket", and for most queries there is nothing relevant. It still has to produce weights that sum to 1. A good way out is to put almost all the weight on one token whose **value vector is harmless** (close to zero), so the head's output is about zero. The first token is the natural place: in a causal model every query can see it, so it is the one position that is always available as a dump.

The paper's Figure 2 shows that trained Llama-2-7B does exactly this: beyond the first two layers, almost all heads put most of their weight on the first token.

![The paper's Figure 2: average attention logits in Llama-2-7B over 256 sentences of length 16, for layers 0, 1, 2, 9, 16, 23 and 31. The first two layers are local (a diagonal band); in all later layers there is a bright column at the first token.](../../assets/architecture_gallery/sink_fig2_attention_logits.png)

Now the arithmetic of what happens when the sink leaves. Take one head, one query, with a sink token holding most of the weight and eight ordinary tokens. Evicting the sink does not remove "a useless token"; it removes most of the **denominator**, and the remaining weights are renormalised to sum to 1 again:

In [2]:
torch.manual_seed(1)
n = 8
logits = torch.randn(n) * 0.5 - 1.0                       # eight ordinary tokens, mildly negative scores
sink_logit = torch.tensor(5.0)                            # the sink gets a large score (it is "easy to attend to")
w_with = torch.softmax(torch.cat([sink_logit[None], logits]), 0)
w_without = torch.softmax(logits, 0)
v = torch.randn(n + 1, 4)                                 # values; the sink's value is tiny (harmless)
v[0] = 0.01 * v[0]
out_with = w_with @ v
out_without = w_without @ v[1:]
print("weight on the sink:", round(w_with[0].item(), 3))
print("weights on the 8 ordinary tokens WITH the sink    :", np.round(w_with[1:].numpy(), 3), " sum =", round(w_with[1:].sum().item(), 3))
print("weights on the 8 ordinary tokens WITHOUT the sink :", np.round(w_without.numpy(), 3), " sum =", round(w_without.sum().item(), 3))
print("output norm with / without the sink:", round(out_with.norm().item(), 3), "/", round(out_without.norm().item(), 3))
check("evicting a sink inflates every remaining weight by 1/(1 - sink weight)",
      torch.allclose(w_without, w_with[1:] / w_with[1:].sum(), atol=1e-6))
check("the head's output changes a lot when the sink is evicted", out_without.norm() > 5 * out_with.norm(),
      f"{out_without.norm():.3f} vs {out_with.norm():.3f}")

weight on the sink: 0.98
weights on the 8 ordinary tokens WITH the sink    : [0.003 0.003 0.003 0.003 0.002 0.002 0.001 0.003]  sum = 0.02
weights on the 8 ordinary tokens WITHOUT the sink : [0.167 0.137 0.124 0.164 0.096 0.111 0.056 0.145]  sum = 1.0
output norm with / without the sink: 0.02 / 1.255
PASS - evicting a sink inflates every remaining weight by 1/(1 - sink weight) 
PASS - the head's output changes a lot when the sink is evicted 1.255 vs 0.020


With the sink the head said "almost nothing" (output norm about 0.1); without it the head says something loud (about 1) because it was forced to spread its entire unit of weight over the ordinary tokens. In a trained model this happens in nearly every head of every layer at once, and the hidden states the next layers receive are no longer anything like what they were trained on. That is the collapse.

## 3. The StreamingLLM cache

The recipe has two parts.

**Part 1: keep the sinks.** The cache holds the first $s$ tokens (the paper uses $s=4$) plus the most recent $w$ tokens. When a new token arrives, the oldest *window* token is evicted; the sinks stay.

**Part 2: positions inside the cache.** With rotary embeddings, the position of a key matters only through relative distance to the query. The paper assigns positions **inside the cache**, not in the original text: if the cache holds sinks 0..3 and the window, the window tokens are placed right after them. So the model never sees a distance larger than $s+w$, however long the text is. That keeps it inside the range it was trained on.

![The paper's Figure 4: the KV cache of StreamingLLM. The first few tokens (the attention sinks) stay in the cache while the rest of the cache rolls forward as new tokens are generated, and older tokens are evicted.](../../assets/architecture_gallery/sink_fig4_kv_cache.png)

Both are implemented in the kit's `Attn.forward(x, window=w, n_sink_tokens=s, cache_pos=True)`. Plain sliding-window attention is `window=w` alone. We show the class again; the new parts are the `_streaming` method and the `window` mask:

In [3]:
class RMSNorm(nn.Module):
    def __init__(self, d, eps=1e-6):
        super().__init__()
        self.w = nn.Parameter(torch.ones(d))
        self.eps = eps

    def forward(self, x):
        return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps) * self.w


def rope_angles(pos, dh, base=10000.0):
    """pos: integer tensor of any shape -> angles [..., dh/2]."""
    inv = base ** (-torch.arange(0, dh, 2, dtype=torch.float32) / dh)
    return pos.to(torch.float32)[..., None] * inv


def apply_rope(x, pos):
    """x: [..., dh] (pairs are (even, odd) channels); pos broadcasts against x[..., 0]."""
    ang = rope_angles(pos, x.shape[-1])
    c, s = ang.cos(), ang.sin()
    x1, x2 = x[..., 0::2], x[..., 1::2]
    out = torch.empty_like(x)
    out[..., 0::2] = x1 * c - x2 * s
    out[..., 1::2] = x1 * s + x2 * c
    return out


def attend(q, k, v, mask, sink=None):
    """Explicit softmax attention.
    q [B,H,T,d]  k,v [B,H,S,d]  mask bool, broadcastable to [B,H,T,S] (True = may look)
    sink: optional [H] learned logit. It is one extra softmax column that has no value vector:
          the probability it takes is simply thrown away (gpt-oss style).
    returns out [B,H,T,d], probs [B,H,T,S] (without the sink column)."""
    B, H, T, d = q.shape
    s = (q @ k.transpose(-1, -2)) / math.sqrt(d)
    s = s.masked_fill(~mask, float("-inf"))
    if sink is not None:
        s = torch.cat([s, sink.view(1, H, 1, 1).expand(B, H, T, 1)], dim=-1)
    p = s.softmax(-1)
    if sink is not None:
        p = p[..., :-1]
    return p @ v, p

class Attn(nn.Module):
    """Causal self-attention with the knobs this series studies.

    n_kv      number of key/value heads (n_kv == n_heads is MHA, 1 is MQA, in between is GQA)
    gate      None | "head" | "elem": sigmoid gate on the attention output, computed from the layer input
    sink      True: one learned sink logit per head
    """
    def __init__(self, d, n_heads, n_kv=None, gate=None, sink=False, rope=True):
        super().__init__()
        n_kv = n_kv or n_heads
        assert n_heads % n_kv == 0
        self.d, self.H, self.Hkv, self.dh = d, n_heads, n_kv, d // n_heads
        self.wq = nn.Linear(d, n_heads * self.dh, bias=False)
        self.wk = nn.Linear(d, n_kv * self.dh, bias=False)
        self.wv = nn.Linear(d, n_kv * self.dh, bias=False)
        self.wo = nn.Linear(n_heads * self.dh, d, bias=False)
        self.gate = gate
        if gate == "elem":
            self.wg = nn.Linear(d, n_heads * self.dh, bias=False)
        elif gate == "head":
            self.wg = nn.Linear(d, n_heads, bias=False)
        self.sink = nn.Parameter(torch.zeros(n_heads)) if sink else None
        self.rope = rope
        self.keep = False                    # set True to store the attention probabilities in self.p
        self.p = None
        self.g = None

    def split(self, x, n):
        B, T, _ = x.shape
        return x.view(B, T, n, self.dh).transpose(1, 2)

    def forward(self, x, window=None, n_sink_tokens=0, cache_pos=False, extra_mask=None):
        """window: keep only the last `window` tokens (sliding-window attention).
        n_sink_tokens: in addition keep the first n tokens forever (StreamingLLM).
        cache_pos: give RoPE the position *inside the cache* rather than the original position.
        extra_mask: bool [B,T,T] (e.g. from DSA's top-k) ANDed onto the causal mask."""
        B, T, _ = x.shape
        q, k, v = self.split(self.wq(x), self.H), self.split(self.wk(x), self.Hkv), self.split(self.wv(x), self.Hkv)
        g = self.H // self.Hkv
        pos = torch.arange(T)
        if window is not None and cache_pos:
            out, p = self._streaming(q, k, v, window, n_sink_tokens)
        else:
            if self.rope:
                q, k = apply_rope(q, pos[None, None, :]), apply_rope(k, pos[None, None, :])
            k, v = k.repeat_interleave(g, 1), v.repeat_interleave(g, 1)
            i, j = pos[:, None], pos[None, :]
            mask = j <= i
            if window is not None:
                mask = mask & ((i - j < window) | (j < n_sink_tokens))
            mask = mask[None, None]
            if extra_mask is not None:
                mask = mask & extra_mask[:, None]
            out, p = attend(q, k, v, mask, self.sink)
        if self.keep:
            self.p = p.detach()
        out = out.transpose(1, 2)                                   # [B,T,H,dh]
        if self.gate is not None:
            gs = torch.sigmoid(self.wg(x))
            self.g = gs.detach() if self.keep else None
            out = out * (gs.view(B, T, self.H, 1) if self.gate == "head" else gs.view(B, T, self.H, self.dh))
        return self.wo(out.reshape(B, T, self.H * self.dh))

    def _streaming(self, q, k, v, window, n_sink):
        """StreamingLLM: query t sees n_sink first tokens + the last `window` tokens, and RoPE uses the
        slot index inside that cache, so the model never sees a distance larger than n_sink+window."""
        B, H, T, dh = q.shape
        S = n_sink + window
        t = torch.arange(T)[:, None]                               # [T,1]
        slot = torch.arange(S)[None, :]                            # [1,S]
        n_cached = torch.clamp(t + 1, max=S)                       # entries in the cache when t is decoded
        # before eviction starts the cache holds tokens 0..t; afterwards sinks + the last `window` tokens
        evicting = (t + 1) > S
        tok = torch.where(evicting & (slot >= n_sink), t - (S - 1 - slot), slot)      # token index held in each slot
        valid = slot < n_cached
        tok = tok.clamp(0, T - 1)
        kg = k[:, :, tok]                                          # [B,Hkv,T,S,dh]
        vg = v[:, :, tok]
        g = H // self.Hkv
        kg, vg = kg.repeat_interleave(g, 1), vg.repeat_interleave(g, 1)
        qpos = (n_cached - 1)[None, None, :, 0]                    # [1,1,T]
        if self.rope:
            q = apply_rope(q, qpos)
            kg = apply_rope(kg, slot[None, None, :, :].expand(1, 1, T, S))
        s = (q[:, :, :, None, :] * kg).sum(-1) / math.sqrt(dh)     # [B,H,T,S]
        s = s.masked_fill(~valid[None, None], float("-inf"))
        if self.sink is not None:
            s = torch.cat([s, self.sink.view(1, H, 1, 1).expand(B, H, T, 1)], -1)
        p = s.softmax(-1)
        if self.sink is not None:
            p = p[..., :-1]
        out = (p[..., None] * vg).sum(3)
        return out, p

In `_streaming`, for every query position $t$ the code builds the list of token indices the cache holds at that moment (`tok`), gathers their keys and values, and rotates keys with the *slot* index (0 to $s+w-1$) and the query with the last slot. A few checks that this does what the paper says:

In [4]:
d, H = 32, 4
m = A.Attn(d, H)
x = torch.randn(2, 40, d)
y_dense = m(x)
check("before anything is evicted, the streaming cache equals dense causal attention", torch.allclose(y_dense, m(x, window=64, cache_pos=True), atol=1e-5))
check("window attention with cache positions equals a masked window (RoPE is relative)", torch.allclose(m(x, window=8), m(x, window=8, cache_pos=True), atol=1e-5))
y_sink = m(x, window=8, n_sink_tokens=2, cache_pos=True)
check("keeping sink tokens changes the output once they would have been evicted", (y_sink[:, 20:] - m(x, window=8, cache_pos=True)[:, 20:]).abs().max() > 1e-3)

# which tokens does the query at position 30 see?
S_, W_ = 2, 8
t = 30
seen = list(range(S_)) + list(range(t - W_ + 1, t + 1))
print(f"query at position {t}, cache = {S_} sinks + window {W_}: sees tokens {seen}")
print(f"RoPE slot positions used for those keys: {list(range(S_ + W_))}  (original distances would be up to {t})")

PASS - before anything is evicted, the streaming cache equals dense causal attention 
PASS - window attention with cache positions equals a masked window (RoPE is relative) 
PASS - keeping sink tokens changes the output once they would have been evicted 
query at position 30, cache = 2 sinks + window 8: sees tokens [0, 1, 23, 24, 25, 26, 27, 28, 29, 30]
RoPE slot positions used for those keys: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]  (original distances would be up to 30)


## 4. A built-in sink: the learned sink logit and softmax-off-by-one

Keeping the first few real tokens is a patch for a habit the model picked up. The paper's own suggestion for future models (their Section 3.3): **give the model a designated sink**, so it does not have to use real tokens. They test two ways, by pre-training 160M-parameter models:

- **Zero sink / softmax-off-by-one**: change the softmax denominator to $1+\sum_i e^{x_i}$, which is the same as prepending a token whose key and value are all zero.
- **Learnable sink token**: prepend one trainable token to every training sequence.

Their Table 3 (perplexity on a PG19 sample, cache config $x+y$ = $x$ initial tokens plus $y$ recent):

| cache | 0+1024 | 1+1023 | 2+1022 | 4+1020 |
|---|---|---|---|---|
| vanilla | 27.87 | 18.49 | 18.05 | 18.05 |
| zero sink | 29214 | 19.90 | 18.27 | 18.01 |
| learnable sink token | 1235 | **18.01** | 18.01 | 18.02 |

The learnable token works with a single initial token. (Zero sink does not fix `0+1024`; the model still used other initial tokens.)

gpt-oss (and MiMo-V2-Flash) use a third form of the same idea, which needs no extra token at all: a **learned sink logit per head**. Add one extra column to the softmax whose score is a learned number $b_h$ and which has no value vector. After the softmax, drop that column. The weights on real tokens now sum to *less* than 1, and the shortfall is what the sink absorbed:

$$ a_j = \frac{e^{x_j}}{e^{b_h} + \sum_i e^{x_i}} \quad\text{(real tokens only)} . $$

Softmax-off-by-one is the special case $b_h=0$ (the zero sink). And since $b_h$ is a parameter of the head, never a token, there is nothing to evict in a sliding window. The kit has it as `Attn(sink=True)`; the `attend` function above implements the extra column. Checks:

In [5]:
q, k, v = torch.randn(1, 1, 3, 8), torch.randn(1, 1, 5, 8), torch.randn(1, 1, 5, 8)
mask = torch.ones(3, 5, dtype=torch.bool)[None, None]
# (1) a sink logit of -infinity is plain softmax
o0, p0 = A.attend(q, k, v, mask)
o1, p1 = A.attend(q, k, v, mask, sink=torch.tensor([-1e9]))
check("sink logit -inf gives ordinary attention", torch.allclose(p0, p1, atol=1e-6))
# (2) sink logit 0 is softmax-off-by-one: e^x / (1 + sum e^x)
s = (q @ k.transpose(-1, -2)) / math.sqrt(8)
off_by_one = s.exp() / (1 + s.exp().sum(-1, keepdim=True))
_, p2 = A.attend(q, k, v, mask, sink=torch.tensor([0.0]))
check("sink logit 0 equals softmax-off-by-one", torch.allclose(p2, off_by_one, atol=1e-6))
# (3) weights sum to less than one; the shortfall is the mass on the sink
_, p3 = A.attend(q, k, v, mask, sink=torch.tensor([3.0]))
print("row sums of the real-token weights with sink logit 3:", np.round(p3.sum(-1).flatten().numpy(), 3))
check("with a positive sink logit the real-token weights sum to less than 1", bool((p3.sum(-1) < 1).all()))

PASS - sink logit -inf gives ordinary attention 
PASS - sink logit 0 equals softmax-off-by-one 
row sums of the real-token weights with sink logit 3: [0.108 0.646 0.155]
PASS - with a positive sink logit the real-token weights sum to less than 1 


## 5. A controlled case: train a head that needs a sink, then evict it

Before any language model, we reproduce the mechanism in the smallest setting that has it, the task from notebook 2: a short list of 10 items (each with a random ID and a random payload), a fixed `BOS` token with zero payload at the start, and a final query token. On *match* queries the target is the matching item's payload. On *no-match* queries the target is zero, so a softmax head needs somewhere harmless to put its weight, and the only zero-payload token is `BOS`. We train one attention layer on this, then **evict `BOS`** at test time (mask it out for every query) and see what happens.

In [6]:
Dm, n_items = 32, 10
def make_task(B, bos, g, p_null=0.5):
    ids = F.normalize(torch.randn(B, n_items, 16, generator=g), dim=-1)
    pay = torch.randn(B, n_items, 16, generator=g)
    items = torch.cat([ids, pay], -1)
    m = torch.randint(0, n_items, (B,), generator=g); ar = torch.arange(B)
    null = torch.rand(B, generator=g) < p_null
    qid = torch.where(null[:, None], F.normalize(torch.randn(B, 16, generator=g), dim=-1), ids[ar, m])
    q = torch.zeros(B, 1, Dm); q[:, 0, :16] = qid
    q[:, 0, 16:] = torch.randn(B, 16, generator=g); q[:, 0, 16] = (~null).float() * 2 - 1       # flag: +1 match, -1 no match
    x = torch.cat([bos.expand(B, 1, Dm), items, q], 1)
    return x, torch.where(null[:, None], torch.zeros(B, 16), pay[ar, m]), null

class Net(nn.Module):
    def __init__(self, sink):
        super().__init__()
        self.att, self.out = A.Attn(Dm, 2, sink=sink, rope=False), nn.Linear(Dm, 16)
    def forward(self, x, mask=None):
        return self.out(self.att(x, extra_mask=mask)[:, -1])

def evict_mask(B, T):
    m = torch.ones(B, T, T, dtype=torch.bool); m[:, :, 0] = False; m[:, 0, 0] = True          # nobody may look at BOS
    return m

def train_net(sink, seed, evict_train, steps=2500):
    torch.manual_seed(seed); g = torch.Generator().manual_seed(seed); bos = torch.randn(1, 1, Dm)
    net = Net(sink); opt = torch.optim.Adam(net.parameters(), 3e-3)
    for s in range(steps):
        x, t, _ = make_task(256, bos, g)
        mask = evict_mask(256, x.shape[1]) if (evict_train and s % 2 == 0) else None            # BOS missing in half the training batches
        loss = F.mse_loss(net(x, mask), t); opt.zero_grad(); loss.backward(); opt.step()
    x, t, nl = make_task(4000, bos, torch.Generator().manual_seed(999))
    with torch.no_grad():
        e_with, e_evict = ((net(x) - t) ** 2).mean(-1), ((net(x, evict_mask(4000, x.shape[1])) - t) ** 2).mean(-1)
    return net, dict(null_with=e_with[nl].mean().item(), null_evict=e_evict[nl].mean().item(),
                     match_with=e_with[~nl].mean().item(), match_evict=e_evict[~nl].mean().item())

t0 = time.time()
plain = [train_net(False, s, False)[1] for s in [0, 1]]
print(f"({time.time() - t0:.0f} s)  plain softmax, trained with BOS always present, 2 seeds:")
for r in plain:
    print(f"   no-match error with BOS {r['null_with']:.4f}  -> BOS evicted {r['null_evict']:.4f};   match error {r['match_with']:.4f} -> {r['match_evict']:.4f}")
check("evicting BOS makes the trained plain head fail on no-match queries (error up by more than 50x)",
      all(r["null_evict"] > 50 * r["null_with"] for r in plain))
check("evicting BOS leaves match queries unchanged (the sink only matters when nothing matches)",
      all(abs(r["match_evict"] - r["match_with"]) < 0.01 for r in plain))

(25 s)  plain softmax, trained with BOS always present, 2 seeds:
   no-match error with BOS 0.0013  -> BOS evicted 0.7840;   match error 0.0682 -> 0.0682
   no-match error with BOS 0.0020  -> BOS evicted 0.8021;   match error 0.0610 -> 0.0610
PASS - evicting BOS makes the trained plain head fail on no-match queries (error up by more than 50x) 
PASS - evicting BOS leaves match queries unchanged (the sink only matters when nothing matches) 


**Result.** The trained plain head solves no-match queries with an error of about 0.001 to 0.002, by putting 98% of its attention on `BOS` (a sink; we measured the same in notebook 2). With `BOS` evicted the error on the same queries is **about 0.8, hundreds of times larger**, while match queries are unaffected. This is the window-attention collapse of the paper's Figure 1, in a layer small enough to read: the head never learned to say "nothing" any other way.

Now the fix. Two heads trained the same way, except that on half of the training batches `BOS` is missing (so the model is forced to find another solution), one with plain softmax, one with a **learned sink logit** per head:

In [7]:
t0 = time.time()
tr_plain = [train_net(False, s, True)[1] for s in [0, 1]]
tr_logit = []
logits_learned = []
for s in [0, 1]:
    net, r = train_net(True, s, True)
    tr_logit.append(r); logits_learned.append(net.att.sink.detach().numpy().round(2))
print(f"({time.time() - t0:.0f} s)  trained with BOS missing on half of the batches, 2 seeds each")
print(f"{'':18s} {'no-match err, BOS there':>24s} {'no-match err, BOS evicted':>27s}")
for name, rs in [("plain softmax", tr_plain), ("learned sink logit", tr_logit)]:
    print(f"{name:18s} {np.mean([r['null_with'] for r in rs]):24.4f} {np.mean([r['null_evict'] for r in rs]):27.4f}")
print("learned sink logits per head:", logits_learned)
check("the learned sink logit lowers the error after eviction compared with plain softmax (same training)",
      np.mean([r["null_evict"] for r in tr_logit]) < 0.6 * np.mean([r["null_evict"] for r in tr_plain]))

(57 s)  trained with BOS missing on half of the batches, 2 seeds each
                    no-match err, BOS there   no-match err, BOS evicted
plain softmax                        0.0096                      0.2738
learned sink logit                   0.0239                      0.1286
learned sink logits per head: [array([5.03, 4.79], dtype=float32), array([4.8 , 4.94], dtype=float32)]
PASS - the learned sink logit lowers the error after eviction compared with plain softmax (same training) 


**Result.** With the eviction in training, the plain head cannot get its no-match error much below 0.27 once `BOS` is gone (there is no zero-payload token left to hide on), while the head with a learned sink logit reaches about 0.13, about half. Its logits grew to about 5, large enough to absorb most of the probability when nothing matches. The fix helps but **does not fully solve** the task here: the error with `BOS` present is a little worse for the sink-logit head (about 0.024 against 0.0095), and 0.13 is far from the 0.001 that the original head reaches with a real `BOS`. One reading, which we did not test: a constant sink logit cannot adapt to how many tokens compete the way a real token with its own key could.

## 6. Experiment: three kinds of toy language model, five cache policies

We reproduce the paper's Section 3.3 comparison at toy scale. Three models, all trained the same way on character-level text with sequences of 64 tokens (4 layers, width 64, 8 heads, rotary embeddings, 600 steps):

1. **vanilla**: ordinary softmax, nothing special.
2. **sink token**: one extra learnable token id placed at the start of every training sequence (the paper's "Sink Token").
3. **sink logit**: `Attn(sink=True)`, the gpt-oss form.

We then evaluate each on a *stream* of 320 characters, longer than the 64 it was trained on, with five cache policies. All policies use a cache budget of 48 tokens in total (smaller than the training length, so the slot positions stay in range):

- **dense**: look at everything (positions go up to 320, beyond what the model saw).
- **window 0+48**: only the last 48 tokens.
- **1+47** and **4+44**: initial tokens kept, plus the window.
- **re-computation**: for each position, restart from scratch on the last 48 tokens (the paper's slow reference).

We report the average loss over stream positions 100 to 319, which is after the first eviction in every case. For the sink-token model, the stream starts with the sink token, and "initial tokens" means the sink token first.

In [8]:
data0 = A.CharData()
V = data0.vocab + 1                      # one extra id is the sink token (unused by the other models)
SINK_ID = data0.vocab
D, H, NL = 64, 8, 4

class SData(A.CharData):
    """Same text; if sink=True every training sequence starts with the sink token."""
    sink = False
    def batch(self, split, B, T, gen):
        s = super().batch(split, B, T, gen)
        if self.sink:
            s = torch.cat([torch.full((B, 1), SINK_ID), s[:, :-1]], 1)
        return s

def build(kind, seed):
    torch.manual_seed(seed)
    att = (lambda i: A.Attn(D, H, sink=True)) if kind == "sink logit" else (lambda i: A.Attn(D, H))
    return A.MiniLM(V, D, att, lambda i: A.SwiGLU(D, 128), n_layers=NL)

kinds = ["vanilla", "sink token", "sink logit"]
models = {}
t0 = time.time()
for kind in kinds:
    for seed in [0, 1]:
        dd = SData(); dd.sink = (kind == "sink token")
        m = build(kind, seed)
        L = A.train_lm(m, dd, steps=600, B=32, T=64, lr=3e-3, seed=seed)
        models[(kind, seed)] = m
        print(f"{kind:11s} seed {seed}: final train loss {np.mean(L[-20:]):.3f}")
print(f"({time.time() - t0:.0f} s)")

vanilla     seed 0: final train loss 1.865


vanilla     seed 1: final train loss 1.819


sink token  seed 0: final train loss 1.881


sink token  seed 1: final train loss 1.861


sink logit  seed 0: final train loss 1.853


sink logit  seed 1: final train loss 1.808
(223 s)


In [9]:
S_LEN, BUDGET, N_STREAMS = 320, 48, 8
val = data0.val
starts = [int(i) for i in torch.linspace(0, len(val) - S_LEN - 2, N_STREAMS)]

def make_streams(kind):
    st = torch.stack([val[s:s + S_LEN + 1] for s in starts])                     # [N, S+1]
    if kind == "sink token":
        st = torch.cat([torch.full((N_STREAMS, 1), SINK_ID), st[:, :-1]], 1)       # prepend the sink token
    return st

@torch.no_grad()
def stream_loss(model, kind, policy, lo=100):
    st = make_streams(kind)
    inp, tgt = st[:, :-1], st[:, 1:]
    model.eval()
    if policy == "dense":
        logits = model(inp)
    elif policy.startswith("window"):
        sinks = int(policy.split()[1].split("+")[0])
        logits = model(inp, window=BUDGET - sinks, n_sink_tokens=sinks, cache_pos=True)
    elif policy == "recompute":
        pos = list(range(lo, inp.shape[1]))
        wins = torch.stack([inp[b, t - BUDGET + 1:t + 1] for b in range(inp.shape[0]) for t in pos])
        last = model(wins)[:, -1]                                                     # [N*len(pos), V]
        ce = F.cross_entropy(last, torch.stack([tgt[b, t] for b in range(inp.shape[0]) for t in pos]))
        model.train(); return ce.item()
    ce = F.cross_entropy(logits[:, lo:].reshape(-1, logits.shape[-1]), tgt[:, lo:].reshape(-1))
    model.train()
    return ce.item()

policies = ["dense", "window 0+48", "window 1+47", "window 4+44", "recompute"]
res = {}
for kind in kinds:
    for pol in policies:
        res[(kind, pol)] = np.mean([stream_loss(models[(kind, s)], kind, pol) for s in [0, 1]])
print(f"{'model':12s}" + "".join(f"{p:>14s}" for p in policies))
for kind in kinds:
    print(f"{kind:12s}" + "".join(f"{res[(kind, p)]:14.3f}" for p in policies))

model                dense   window 0+48   window 1+47   window 4+44     recompute
vanilla              2.899         2.475         2.476         2.477         2.473
sink token           3.277         2.493         2.489         2.488         2.489
sink logit           2.901         2.461         2.461         2.463         2.461


**Result (mean of 2 seeds; loss in nats on stream positions 100 to 319; the table above).** **Nothing collapsed.** Window attention with no initial tokens kept (0+48) matches the re-computation baseline in all three models (2.475 against 2.473 for vanilla, 2.493 against 2.489 for the sink-token model, 2.461 against 2.461 for the sink-logit model), and keeping 1 or 4 initial tokens changes the loss by at most 0.003, which is noise. The policy that fails is **dense attention**: 2.90 for vanilla and 3.28 for the sink-token model, against about 2.47 with a window, because the stream is 320 tokens long and the models only ever saw positions up to 64, so rotary angles at large distances are outside what they learned. That is the paper's first observation (Figure 3: dense attention fails once the text is longer than the training length), reproduced. The paper's second observation, that window attention collapses when the first tokens leave, is **not reproduced**: as the next measurement shows, these models have no sink to lose. The sink-token model is slightly worse than vanilla under every policy (and much worse under dense), so putting one token at the start of every sequence did not help here.

### Does a sink actually exist in these models?

The earlier notebook found no first-token sink in a toy. We measure it again, now in the three models trained here, as the share of attention (all layers, all heads, queries from position 8 on) that goes to position 0 on 64-token validation text. For the sink-token model, position 0 is the sink token itself.

In [10]:
@torch.no_grad()
def share_first(model, kind, n_batches=6):
    dd = SData(); dd.sink = (kind == "sink token")
    g = torch.Generator().manual_seed(11)
    for b in model.blocks: b.attn.keep = True
    model.eval()
    sh = torch.zeros(NL)
    for _ in range(n_batches):
        seq = dd.batch("val", 16, 64, g)
        model(seq[:, :-1])
        for l, b in enumerate(model.blocks):
            sh[l] += b.attn.p[:, :, 8:, 0].mean().item() / n_batches
    for b in model.blocks: b.attn.keep = False
    model.train()
    return sh

shares = {k: np.mean([share_first(models[(k, s)], k).numpy() for s in [0, 1]], 0) for k in kinds}
for k, v in shares.items():
    print(f"{k:11s} per layer {np.round(v, 3)}   mean {v.mean():.3f}")
# the sink-logit model: how much weight goes to the sink column?
@torch.no_grad()
def sink_logit_mass(model):
    out = []
    g = torch.Generator().manual_seed(11)
    seq = SData().batch("val", 16, 64, g)
    for b in model.blocks: b.attn.keep = True
    model.eval(); model(seq[:, :-1]); model.train()
    for b in model.blocks:
        out.append(1 - b.attn.p[:, :, 8:].sum(-1).mean().item()); b.attn.keep = False
    return np.array(out)
mass = np.mean([sink_logit_mass(models[("sink logit", s)]) for s in [0, 1]], 0)
print(f"sink-logit model: weight absorbed by the sink column, per layer {np.round(mass, 3)}   mean {mass.mean():.3f}")
print("learned sink logits (layer 1, 8 heads):", np.round(models[("sink logit", 0)].blocks[0].attn.sink.detach().numpy(), 2))

vanilla     per layer [0.032 0.011 0.009 0.005]   mean 0.014
sink token  per layer [0.042 0.012 0.007 0.009]   mean 0.017
sink logit  per layer [0.032 0.01  0.012 0.007]   mean 0.015
sink-logit model: weight absorbed by the sink column, per layer [0.03  0.023 0.027 0.032]   mean 0.028
learned sink logits (layer 1, 8 heads): [0.17 0.4  0.18 0.22 0.35 0.3  0.39 0.37]


**Result.** The first token gets 0.5% to 4% of the attention in every layer of all three models (means 1.4%, 1.7%, 1.5%), the same as the toy of notebook 2: **no attention sink formed**, so nothing could be evicted. Even the dedicated sink token of the sink-token model gets only 1.7% on average, and the learned sink logits of the third model stay small (0.2 to 0.4) and absorb about 3% of the weight. We did not investigate why a sink does not form in this setting; the paper's evidence is from models of 160M parameters and up, trained on billions of tokens. What we have are two separate demonstrations: the mechanism in section 5, and the absence of the problem in a small language model trained briefly in section 6.

## 7. Local and global layers: the structural fix

A third approach does not touch the softmax at all. Make most layers **local** (each token sees only the last $w$ tokens) and only an occasional layer **global** (sees everything). Gemma 3 uses five local layers for every global one; Trinity Large uses three local per global; gpt-oss alternates layers with a 128-token window; Muse Spark Open uses rotary embeddings only in its local layers (the diagram's note). The point is the cache: a local layer never needs more than $w$ cached tokens, so the cache cost is dominated by the few global layers.

For $L$ layers with a fraction $f$ of global layers, window $w$, and a conversation of $S$ tokens, cached tokens per head-slot are $L\,[f\,S + (1-f)\min(S,w)]$. As an **illustration** (an invented configuration, not any real model's): 60 layers, window 1,024, one global layer in six:

In [11]:
L_, w_, f_ = 60, 1024, 1 / 6
print(f"{'conversation length':>20s} {'all global':>12s} {'5:1 local/global':>18s} {'saving':>8s}")
for S in [1024, 8192, 32768, 131072]:
    allg = L_ * S
    mix = L_ * (f_ * S + (1 - f_) * min(S, w_))
    print(f"{S:20,d} {allg:12,d} {mix:18,.0f} {allg / mix:7.1f}x")
S = 131072
check("with 1/6 global layers the cache can never shrink by more than 6x (the global layers dominate)",
      L_ * S / (L_ * (f_ * S + (1 - f_) * w_)) < 6.0)
check("short conversations see no saving (everything fits in the window)", L_ * 1024 == L_ * (f_ * 1024 + (1 - f_) * 1024))

 conversation length   all global   5:1 local/global   saving
               1,024       61,440             61,440     1.0x
               8,192      491,520            133,120     3.7x
              32,768    1,966,080            378,880     5.2x
             131,072    7,864,320          1,361,920     5.8x
PASS - with 1/6 global layers the cache can never shrink by more than 6x (the global layers dominate) 
PASS - short conversations see no saving (everything fits in the window) 


The saving tends to $1/f$ for long conversations and never beats it: one global layer in six means at most a 6x smaller cache. That is why this scheme is usually *combined* with grouped-query attention (notebook 1) rather than replacing it.

A common worry is that a local layer cannot "see" beyond its window. That is true for one layer, but layers stack: information can hop. After $\ell$ local layers with window $w$, a token's output can depend on tokens up to $\ell\,(w-1)$ positions back. We confirm with a one-line experiment: perturb the first token and see in which positions of a 6-layer window-4 model the output changes.

In [12]:
torch.manual_seed(0)
dd = 32
nl, ww = 6, 4
layers = nn.ModuleList([A.Attn(dd, 4) for _ in range(nl)]).double()        # double precision: the far-away effects are tiny
def run(xx):
    h = xx
    for l in layers:
        h = h + l(F.layer_norm(h, (dd,)), window=ww)
    return h
x = torch.randn(1, 40, dd, dtype=torch.float64)
x2 = x.clone(); x2[:, 0] += 3.0 * torch.randn(dd, dtype=torch.float64)          # a random direction (a constant shift would be removed by layer_norm)
diff = (run(x) - run(x2)).abs().amax(-1)[0]
reach = int((diff > 1e-13).nonzero().max())
print("positions affected by a change at position 0:", [int(i) for i in (diff > 1e-13).nonzero().flatten()])
print(f"last affected position {reach};  theory: (window-1) x layers = {(ww - 1) * nl}")
check("the reach of stacked local layers is (window - 1) x layers", reach == (ww - 1) * nl)

positions affected by a change at position 0: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18]
last affected position 18;  theory: (window-1) x layers = 18
PASS - the reach of stacked local layers is (window - 1) x layers 


So six layers of window 4 reach 18 tokens back, and a 1,024-token window over 50 local layers reaches about 50,000. That reach is in principle available, though the paper (and everyone since) points out that information passed through many hops is lossy, and it is the global layers that provide exact long-range lookup.

## What this notebook did and did not show

**Shown (checks in the cells).** Evicting a high-weight sink inflates every remaining attention weight by $1/(1-a_{sink})$ and changes the head's output by a large factor, which is the mechanism behind window attention's collapse. The StreamingLLM cache (sinks plus window, RoPE slots inside the cache) is exactly dense attention before eviction and exactly a windowed attention when no sinks are kept. A learned sink logit of $-\infty$ gives plain softmax, of 0 gives softmax-off-by-one, and a positive one absorbs weight. Stacked local layers reach $(w-1)\ell$ tokens. A mix with one global layer in six cannot cut the cache by more than 6x.

**Experiments.** Section 5 builds a head that needs a sink, evicts the sink and measures the damage, with and without a learned sink logit. Section 6's table and the measurement after it report what three toy language models did under five cache policies; the interpretation is written under each from the printed numbers.

**Not shown.** The paper's headline results (stable perplexity to 4 million tokens on Llama-2, MPT, Falcon and Pythia; 22.2x speed-up over re-computation) are for 7B to 70B pre-trained models and need real inference kernels; we have neither. Our streams are 320 characters and our models 0.2M parameters, so the failure that matters to the paper (a sink that has formed over trillions of tokens) may not have formed in ours. The cache-size table is arithmetic for an invented configuration. We did not test softmax-off-by-one as a trained model.

**Next.** Notebook 4 is about the opposite problem: keep the full context but attend to only a few of its tokens, chosen by a tiny scoring network (DeepSeek Sparse Attention).

In [13]:
print("passed:", len(PASS), "| failed:", len(FAIL))
assert not FAIL, FAIL

passed: 14 | failed: 0
